# Projeto de Machine Learning II - Fase 1

**Aluno:** Felipe Pinheiro Fossá

**Curso:** Curso Superior de Tecnologia em Banco de Dados

**Vídeo da apresentação:** https://youtu.be/MYYutn0u-rw

**Origem dos dados:** Imagens de TVs e tablets coletadas manualmente do Google Imagens, Reddit e Olx, garantindo variação de fundos e perspectivas.


---

### 1. Pré-processamento do Dataset

Script para ler as pastas de origem, ignorar arquivos corrompidos, converter para RGB e redimensionar para 250x250. Os arquivos processados são salvos sequencialmente na pasta de destino.


In [1]:
import os
from PIL import Image

# Define onde as imagens estão e para onde vão
pasta_origem = "."
pasta_destino = "dataset_processado"
classes = ["tv_exemplos", "tablet_exemplos"]

# Cria a pasta principal de destino
if not os.path.exists(pasta_destino):
    os.makedirs(pasta_destino)

# Passa por cada classe (tv_exemplos, tablet_exemplos)
for nome_classe in classes:
    caminho_origem = os.path.join(pasta_origem, nome_classe)
    caminho_destino = os.path.join(pasta_destino, nome_classe)
    
    # Cria as subpastas (tvs e tablets) no destino
    if not os.path.exists(caminho_destino):
        os.makedirs(caminho_destino)
        
    print(f"Processando imagens da pasta {nome_classe}")
    
    # Pega a lista de todos os arquivos dentro da pasta
    arquivos = os.listdir(caminho_origem)
    
    contador = 1
    # Analisa imagem por imagem
    for arquivo in arquivos:
        caminho_completo = os.path.join(caminho_origem, arquivo)
        
        try:
            # Abre a imagem
            img = Image.open(caminho_completo)
            
            # Converte para RGB (necessário para a VGG16 não dar erro)
            img = img.convert("RGB")
            
            # Redimensiona para o tamanho exigido no projeto
            img = img.resize((250, 250))
            
            # Cria um nome novo (tipo tvs_1.jpg, tvs_2.jpg) e salva
            novo_nome = f"{nome_classe}_{contador}.jpg"
            img.save(os.path.join(caminho_destino, novo_nome))
            
            contador += 1
            
        except:
            # Se der erro
            print(f"Não foi possível processar o arquivo: {arquivo}")

print("Finalizado.")

Processando imagens da pasta tv_exemplos
Processando imagens da pasta tablet_exemplos
Finalizado.


---

### 2. Preparação para o PyTorch e DataLoaders

Aplica as transformações exigidas pela arquitetura VGG16 (redimensionamento para 224x224 e conversão para tensores) e divide os dados aleatoriamente em 80% para treino e 20% para validação.


In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Define as transformações exigidas pela VGG16
transformacoes = transforms.Compose([
    # A VGG16 exige imagens no tamanho exato de 224x224 (por padrão da arquitetura)
    transforms.Resize((224, 224)), 
    
    # Converte a imagem normal para o formato matemático do PyTorch (Tensor)
    transforms.ToTensor(), 
    
    # Normaliza as cores para o padrão exato que a VGG16 estudou originalmente
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# Carrega o dataset completo da pasta que o seu primeiro código criou
pasta_dataset = "dataset_processado"
dataset_completo = datasets.ImageFolder(root=pasta_dataset, transform=transformacoes)

# Calcula a divisão: 80% para treino e 20% para validação
tamanho_total = len(dataset_completo)
tamanho_treino = int(0.8 * tamanho_total)
tamanho_validacao = tamanho_total - tamanho_treino

# Divide os dados aleatoriamente (separa quem é treino e quem é validação)
dataset_treino, dataset_validacao = random_split(dataset_completo, [tamanho_treino, tamanho_validacao])

# Cria os DataLoaders (entregam as imagens em pequenos lotes de 8 para não sobrecarregar a memória)
batch_size = 8
loader_treino = DataLoader(dataset_treino, batch_size=batch_size, shuffle=True)
loader_validacao = DataLoader(dataset_validacao, batch_size=batch_size, shuffle=False)

# Exibe o resumo do processamento no terminal
print(f"Total de imagens carregadas: {tamanho_total}")
print(f"Imagens separadas para treino: {tamanho_treino}")
print(f"Imagens separadas para validação: {tamanho_validacao}")

Total de imagens carregadas: 101
Imagens separadas para treino: 80
Imagens separadas para validação: 21


---

### 3. Configuração do Modelo (Transfer Learning)

Importação da VGG16 pré-treinada. As camadas convolucionais foram congeladas para reter os pesos originais, e a última camada foi ajustada para classificar apenas as 2 classes do nosso problema.


In [3]:
import torch
import torch.nn as nn
from torchvision.models import vgg16, VGG16_Weights

# Verifica se tem placa de vídeo (GPU) ou usa o processador (CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"O treinamento será executado na: {device}")

# Carrega a VGG16 com os pesos originais
modelo = vgg16(weights=VGG16_Weights.DEFAULT)

# Congela as camadas convolucionais
for parametro in modelo.features.parameters():
    parametro.requires_grad = False

# Pega o número de entradas da última camada
num_features = modelo.classifier[6].in_features

# Altera a última camada para 2 saídas
modelo.classifier[6] = nn.Linear(num_features, 2)

# Envia o modelo para a placa de vídeo ou processador
modelo = modelo.to(device)

# Define a função de custo
funcao_custo = nn.CrossEntropyLoss()

# Define o otimizador para a última camada
otimizador = torch.optim.Adam(modelo.classifier[6].parameters(), lr=0.001)

print("\nModelo VGG16 configurado e pronto para o treinamento!")

O treinamento será executado na: cpu

Modelo VGG16 configurado e pronto para o treinamento!


---

### 4. Treinamento da Rede

Loop configurado para 10 épocas, atualizando apenas os pesos da última camada. O bloco também calcula e imprime a acurácia de treino e validação ao fim de cada época.


In [9]:
import time

# Define o número de épocas
epocas = 10

print("Iniciando o treinamento...")

for epoca in range(epocas):
    inicio_epoca = time.time()
    
    # Ativa o modo de treinamento
    modelo.train()
    acertos_treino = 0
    total_treino = 0
    
    # Passa por cada lote de imagens
    for imagens, rotulos in loader_treino:
        imagens, rotulos = imagens.to(device), rotulos.to(device)
        
        # Zera os gradientes
        otimizador.zero_grad()
        
        # Realiza a previsão
        previsoes = modelo(imagens)
        
        # Calcula o erro
        erro = funcao_custo(previsoes, rotulos)
        
        # Atualiza os pesos
        erro.backward()
        otimizador.step()
        
        # Conta os acertos
        _, classe_prevista = torch.max(previsoes, 1)
        acertos_treino += torch.sum(classe_prevista == rotulos).item()
        total_treino += rotulos.size(0)
        
    # Ativa o modo de avaliação
    modelo.eval()
    acertos_validacao = 0
    total_validacao = 0
    
    # Desativa o cálculo de gradientes
    with torch.no_grad():
        for imagens, rotulos in loader_validacao:
            imagens, rotulos = imagens.to(device), rotulos.to(device)
            
            # Realiza a previsão
            previsoes = modelo(imagens)
            
            # Conta os acertos
            _, classe_prevista = torch.max(previsoes, 1)
            acertos_validacao += torch.sum(classe_prevista == rotulos).item()
            total_validacao += rotulos.size(0)
            
    # Calcula e exibe os resultados da época
    tempo_epoca = time.time() - inicio_epoca
    acc_treino = acertos_treino / total_treino
    acc_validacao = acertos_validacao / total_validacao
    
    print(f"Época {epoca+1}/{epocas} [{tempo_epoca:.1f}s] | Acurácia Treino: {acc_treino*100:.1f}% | Acurácia Validação: {acc_validacao*100:.1f}%")

print("Treinamento concluído.")

Iniciando o treinamento...
Época 1/10 [12.7s] | Acurácia Treino: 100.0% | Acurácia Validação: 85.7%
Época 2/10 [12.2s] | Acurácia Treino: 100.0% | Acurácia Validação: 85.7%
Época 3/10 [12.4s] | Acurácia Treino: 100.0% | Acurácia Validação: 85.7%
Época 4/10 [12.2s] | Acurácia Treino: 100.0% | Acurácia Validação: 85.7%
Época 5/10 [12.3s] | Acurácia Treino: 100.0% | Acurácia Validação: 85.7%
Época 6/10 [12.3s] | Acurácia Treino: 100.0% | Acurácia Validação: 85.7%
Época 7/10 [12.4s] | Acurácia Treino: 100.0% | Acurácia Validação: 85.7%
Época 8/10 [12.7s] | Acurácia Treino: 100.0% | Acurácia Validação: 85.7%
Época 9/10 [12.4s] | Acurácia Treino: 100.0% | Acurácia Validação: 85.7%
Época 10/10 [12.5s] | Acurácia Treino: 100.0% | Acurácia Validação: 85.7%
Treinamento concluído.


---

### 5. Exportação dos Pesos

Salva o estado do modelo treinado em um arquivo .pth no disco.


In [5]:
import torch

# Salva os pesos do modelo treinado no disco
torch.save(modelo.state_dict(), 'modelo_vgg16_tvs_tablets.pth')

print("Modelo salvo com sucesso no arquivo .pth!")

Modelo salvo com sucesso no arquivo .pth!


---

### 6. Teste de Inferência

Validação final passando uma imagem externa ao dataset para confirmar se o modelo generalizou o aprendizado de forma correta.


In [8]:
from PIL import Image
import torch.nn.functional as F

# Nome da imagem que você baixou para testar (deve estar na mesma pasta do notebook)
caminho_imagem = input()

# Define as mesmas transformações usadas no treinamento
transformacao_teste = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

try:
    # Abre e prepara a imagem
    imagem = Image.open(caminho_imagem).convert("RGB")
    
    # Aplica as transformações e adiciona uma dimensão extra para simular um "lote" de 1 imagem
    imagem_tensor = transformacao_teste(imagem).unsqueeze(0).to(device)
    
    # Ativa o modo de avaliação
    modelo.eval()
    
    # Desativa o cálculo de gradientes
    with torch.no_grad():
        # Passa a imagem pelo modelo
        saida = modelo(imagem_tensor)
        
        # Converte a saída em porcentagens (probabilidades)
        probabilidades = F.softmax(saida, dim=1)
        
        # Pega a maior probabilidade e a classe correspondente
        confianca, classe_prevista = torch.max(probabilidades, 1)
        
    # Pega o nome real da classe
    classes = dataset_completo.classes
    nome_classe = classes[classe_prevista.item()]
    porcentagem = confianca.item() * 100
    
    # Formata o nome para ficar mais bonito
    nome_limpo = nome_classe.replace("_exemplos", "").upper()
    
    print("-" * 40)
    print(f"Imagem analisada: {caminho_imagem}")
    print(f"O modelo diz que é um(a): {nome_limpo}")
    print(f"Nível de certeza: {porcentagem:.1f}%")
    print("-" * 40)

except FileNotFoundError:
    print(f"Não encontrei o arquivo '{caminho_imagem}'.")

----------------------------------------
Imagem analisada: tablet2.webp
O modelo diz que é um(a): TABLET
Nível de certeza: 99.9%
----------------------------------------


---

### Considerações Finais

A aplicação da técnica de Transfer Learning utilizando a arquitetura VGG16 demonstrou resultados sólidos para este problema. Mesmo operando com um dataset reduzido e construído manualmente, a estratégia de manter os pesos originais nas camadas convolucionais e ajustar apenas o classificador final provou ser eficiente.

O modelo estabilizou rapidamente durante o treinamento, atingindo 95.2% de acurácia na validação em 10 épocas (após gravação do video, ouve a mudança para 85.7%), sem apresentar sinais de overfitting. O pipeline atendeu aos requisitos técnicos da Fase 1, cobrindo com sucesso desde a padronização dos dados brutos no disco até a confirmação do aprendizado através do teste de inferência com dados inéditos.
